In [2]:
import cv2
import numpy as np
import glob
import os
ESCALA_PROC = 1200
AREA_MIN_PX = 1000
AREA_MAX_PX = 150000
SOLIDEZ_MIN = 0.50
V_BLANCO_MIN = 155
S_PUNTO_MAX = 58
ROI_NORM_SIZE = 120
def segmentar_dados(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask_sat = cv2.inRange(hsv, np.array([0, 50, 40]), np.array([180, 255, 255]))
    mask_dark = cv2.inRange(hsv, np.array([0, 0, 0]), np.array([180, 80, 70]))
    mask = cv2.bitwise_or(mask_sat, mask_dark)
    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    return mask
def recortar_dado_recto(img_bgr, contorno, margen_rel=0.25):
    rect = cv2.minAreaRect(contorno)
    (cx, cy), (rw, rh), angle = rect
    if rw < rh:
        rw, rh = rh, rw
        angle += 90
    rw_m = rw * (1.0 + margen_rel)
    rh_m = rh * (1.0 + margen_rel)
    H, W = img_bgr.shape[:2]
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    img_rot = cv2.warpAffine(
        img_bgr, M, (W, H),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255, 255, 255),
    )
    x1 = max(0, int(cx - rw_m / 2))
    y1 = max(0, int(cy - rh_m / 2))
    x2 = min(W, int(cx + rw_m / 2))
    y2 = min(H, int(cy + rh_m / 2))
    return img_rot[y1:y2, x1:x2]
def tipo_dado(roi_bgr):
    h, w = roi_bgr.shape[:2]
    hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    cy, cx = h // 2, w // 2
    mg = max(4, min(14, w // 6, h // 6))
    zona = hsv[max(0, cy - mg):cy + mg, max(0, cx - mg):cx + mg]
    s_m = float(np.mean(zona[:, :, 1]))
    v_m = float(np.mean(zona[:, :, 2]))
    if v_m < 100: return 'oscuro'
    elif s_m > 40: return 'saturado'
    else: return 'claro'
def contar_puntos_topologico(roi_norm, tipo):
    rh, rw = roi_norm.shape[:2]
    hsv = cv2.cvtColor(roi_norm, cv2.COLOR_BGR2HSV)
    if tipo == 'saturado':
        m_pts = cv2.inRange(hsv, np.array([0, 0, V_BLANCO_MIN]), np.array([180, S_PUNTO_MAX, 255]))
    elif tipo == 'oscuro':
        m_blanco = cv2.inRange(hsv, np.array([0, 0, 130]), np.array([180, 58, 255]))
        m_color = cv2.inRange(hsv, np.array([0, 90, 50]), np.array([180, 255, 200]))
        m_pts = m_blanco if int(cv2.countNonZero(m_blanco)) >= int(cv2.countNonZero(m_color)) else m_color
    else:
        gray = cv2.cvtColor(roi_norm, cv2.COLOR_BGR2GRAY)
        _, m_pts = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    k2 = np.ones((2, 2), np.uint8)
    m_pts = cv2.morphologyEx(m_pts, cv2.MORPH_OPEN, k2)
    n_comp, labels, stats, _ = cv2.connectedComponentsWithStats(m_pts, 8, cv2.CV_32S)
    borde_labels = set()
    borde_labels.update(labels[0, :].tolist())
    borde_labels.update(labels[-1, :].tolist())
    borde_labels.update(labels[:, 0].tolist())
    borde_labels.update(labels[:, -1].tolist())
    area_roi = rh * rw
    min_a = max(6, area_roi * 0.005)
    max_a = area_roi * 0.18
    puntos = 0
    for j in range(1, n_comp):
        if j in borde_labels: continue
        ap = int(stats[j, cv2.CC_STAT_AREA])
        wp_c = int(stats[j, cv2.CC_STAT_WIDTH])
        hp_c = int(stats[j, cv2.CC_STAT_HEIGHT])
        if min_a < ap < max_a and 0.3 < (wp_c / max(hp_c, 1)) < 3.2:
            puntos += 1
    return puntos
def contar_puntos_hough(roi_bgr, tipo):
    roi = cv2.resize(roi_bgr, (ROI_NORM_SIZE, ROI_NORM_SIZE))
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    if tipo == 'saturado':
        m = cv2.inRange(hsv, np.array([0, 0, V_BLANCO_MIN]), np.array([180, S_PUNTO_MAX, 255]))
    elif tipo == 'oscuro':
        m_b = cv2.inRange(hsv, np.array([0, 0, 130]), np.array([180, 58, 255]))
        m_c = cv2.inRange(hsv, np.array([0, 90, 50]), np.array([180, 255, 200]))
        m = m_b if cv2.countNonZero(m_b) >= cv2.countNonZero(m_c) else m_c
    else:
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        _, m = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    k2 = np.ones((2, 2), np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k2)
    blur = cv2.GaussianBlur(m, (3, 3), 0)
    circs = cv2.HoughCircles(
        blur, cv2.HOUGH_GRADIENT, dp=1, minDist=10,
        param1=50, param2=10, minRadius=3, maxRadius=22,
    )
    if circs is not None:
        n = len(circs[0])
        if 1 <= n <= 6: return n
    return 0
def procesar_imagen(ruta_imagen, guardar=False, carpeta_salida="resultados", mostrar=False):
    arr = np.fromfile(ruta_imagen, np.uint8)
    img_orig = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img_orig is None:
        print(f"[ERROR] No se pudo leer: {ruta_imagen}")
        return 0, []
    h, w = img_orig.shape[:2]
    escala = ESCALA_PROC / max(h, w)
    img = cv2.resize(img_orig, (int(w * escala), int(h * escala)))
    mask = segmentar_dados(img)
    contornos, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lista_puntos = []
    img_resultado = img.copy() if (guardar or mostrar) else None
    for c in contornos:
        area = cv2.contourArea(c)
        if not (AREA_MIN_PX < area < AREA_MAX_PX): continue
        hull_area = cv2.contourArea(cv2.convexHull(c))
        if hull_area == 0 or (area / hull_area) < SOLIDEZ_MIN: continue
        x, y, bw, bh = cv2.boundingRect(c)
        if not (0.40 < bw / bh < 2.50): continue
        roi_bgr = recortar_dado_recto(img, c, margen_rel=0.25)
        if roi_bgr.shape[0] < 20 or roi_bgr.shape[1] < 20: continue
        t = tipo_dado(roi_bgr)
        roi_norm = cv2.resize(roi_bgr, (ROI_NORM_SIZE, ROI_NORM_SIZE))
        pts = contar_puntos_topologico(roi_norm, t)
        if pts == 0: pts = contar_puntos_hough(roi_bgr, t)
        if 1 <= pts <= 6:
            lista_puntos.append(pts)
            if img_resultado is not None:
                cv2.rectangle(img_resultado, (x, y), (x + bw, y + bh), (0, 220, 50), 2)
                cv2.putText(img_resultado, str(pts), (x + 4, y + 28), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 220), 2, cv2.LINE_AA)
                cv2.putText(img_resultado, t[:3], (x + 4, y + bh - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (60, 60, 60), 1)
    lista_puntos.sort()
    if guardar and img_resultado is not None:
        os.makedirs(carpeta_salida, exist_ok=True)
        base = os.path.basename(ruta_imagen)
        salida = os.path.join(carpeta_salida, f"res_{base}")
        cv2.imwrite(salida, img_resultado)
    if mostrar and img_resultado is not None:
        cv2.imshow(f"Procesando: {os.path.basename(ruta_imagen)}", img_resultado)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    return len(lista_puntos), lista_puntos
PATRON = "img_colores/*.jpg"
GUARDAR = True
CARPETA_SALIDA = "resultados_colores"
MOSTRAR = False
archivos = sorted(glob.glob(PATRON))
if not archivos:
    print(f"No se encontraron imágenes con el patrón: {PATRON}")
else:
    sep = "─" * 65
    print(sep)
    print(f"Procesando {len(archivos)} imagen(es) | Guardar: {GUARDAR} en '{CARPETA_SALIDA}'")
    print(sep)
    total_dados = 0
    suma_total = 0
    for archivo in archivos:
        nombre = os.path.basename(archivo)
        n, caras = procesar_imagen(archivo, guardar=GUARDAR, carpeta_salida=CARPETA_SALIDA, mostrar=MOSTRAR)
        total_dados += n
        suma_total += sum(caras)
        print(f"{nombre:<42s} dados:{n:2d} puntos:{str(caras):<20s} suma:{sum(caras)}")
    print(sep)
    print(f"Total dados : {total_dados}")
    print(f"Suma puntos : {suma_total}")
    print(sep)

─────────────────────────────────────────────────────────────────
Procesando 8 imagen(es) | Guardar: True en 'resultados_colores'
─────────────────────────────────────────────────────────────────
IMG_20191209_100614.jpg                    dados: 2 puntos:[1, 3]               suma:4
IMG_20191209_100620.jpg                    dados: 4 puntos:[1, 3, 3, 6]         suma:13
IMG_20191209_100625.jpg                    dados: 4 puntos:[1, 3, 3, 3]         suma:10
IMG_20191209_100629.jpg                    dados: 4 puntos:[2, 3, 5, 5]         suma:15
IMG_20191209_100633.jpg                    dados: 4 puntos:[2, 3, 3, 5]         suma:13
IMG_20191209_100641.jpg                    dados: 6 puntos:[1, 2, 2, 4, 5, 6]   suma:20
IMG_20191209_100645.jpg                    dados: 6 puntos:[1, 2, 2, 3, 4, 5]   suma:17
IMG_20191209_100649.jpg                    dados: 4 puntos:[1, 1, 3, 4]         suma:9
─────────────────────────────────────────────────────────────────
Total dados : 34
Suma puntos : 101
─